In [0]:
%run "../03.Silver/00.Common"

In [0]:
def read_silver_traffic():
    df = (
        spark.readStream
            .table('traffic_data.silver.trafficload')
    )
    print('read silver traffic')
    return df

In [0]:
def read_silver_roads():
    df = (
        spark.readStream
            .table('traffic_data.silver.roadsload')
    )
    print('read silver roads')
    return df

In [0]:
def vehicle_intencity(df):
    from pyspark.sql import functions as F
    df_clean = (df.withColumn('Vehicle_Intencity',  F.try_divide(F.col('Motor_vehicle_count'),F.col('Link_length_km'))))
    print('vehicle intencity Column added')
    return df_clean

In [0]:
def load_time(df):
    from pyspark.sql import functions as F
    df_clean = (df.withColumn('Load_Time', F.current_timestamp()))
    print('Load Time Column added')
    return df_clean


In [0]:
def write_traffic_to_gold(df):
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    print('write to gold traffic started')
    ws = (df.writeStream
               .format('delta')
               .queryName('GoldTraffic')
               .option('CheckpointLocation', f'{source}/Checkpoint/Gold/TrafficLoad')
               .trigger(availableNow = True)
               .outputMode('append')
               .toTable('traffic_data.gold.trafficload')
            )
    ws.awaitTermination()
    print('write to gold traffic completed')

In [0]:
def write_roads_to_gold(df):
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    print('write to gold roads started')
    ws = (df.writeStream
               .format('delta')
               .queryName('GoldRoads')
               .option('CheckpointLocation', f'{source}/Checkpoint/Gold/RoadLoad')
               .trigger(availableNow = True)
               .outputMode('append')
               .toTable('traffic_data.gold.roadsload')
            )
    ws.awaitTermination()
    print('write to gold roads completed')

In [0]:
df_traffic = read_silver_traffic()
df_roads = read_silver_roads()


df1 = vehicle_intencity(df_traffic)
df2 = load_time(df1)

df3 = load_time(df_roads)

write_traffic_to_gold(df2)
write_roads_to_gold(df3)


In [0]:
%sql 
select * from traffic_data.gold.trafficload

In [0]:
%sql
select * from traffic_data.gold.roadsload